In [19]:
import yaml
from types import SimpleNamespace
from strategies.run_allen import get_allen_result, get_allen_signals
from strategies.run_gino import get_gino_result, get_gino_signals
from utils.io import read_all_df
import pandas as pd
from evaluation.stats import sharpe

CATEGGORY = '汽車'
result_dir = "results_dir/CarTSE"

def load_config(path):
    with open(path, "r") as f:
        cfg_dict = yaml.safe_load(f)
    return SimpleNamespace(**cfg_dict)
backtest_config = load_config("config/backtest.yaml")

f = open("config/backtest.yaml")
cfg = yaml.safe_load(f)

result = get_allen_result(cfg, result_dir)
signals = get_allen_signals(cfg, result_dir)
test_pred, test_truth, _, _ = read_all_df(result_dir)

輸入成功!


In [20]:
import os
import finlab
from finlab import data
finlab.login('ntSS3778pZi2FfkeYxXP0p+S0iI4AggkcphAUxh/lTVrWqT2FreKQsDkTA92CM7d#vip_m')
dir = os.listdir(result_dir)[0]
pred_df = pd.read_csv(os.path.join(result_dir, dir, "test/pred_pct.csv"), index_col="date")
close_price = data.get('price:收盤價')[pred_df.columns]
close_price = close_price[(close_price.index >= "2021-01-01") & (close_price.index <= "2024-12-31")]
for col in close_price.columns:
    close_price[col] = close_price[col] / close_price[col].iloc[0]
baseline_returns = close_price.mean(axis=1)

輸入成功!


In [21]:
def yearly_returns(returns):
    yearly_roi = returns.resample("Y").last() / returns.resample("Y").first() - 1
    rois = {}
    for year in yearly_roi.index:
        rois[year.year] = yearly_roi[year]
    rois['final'] = returns.iloc[-1] / returns.iloc[0] - 1
    rois['sharpe'] = sharpe(returns)
    return rois

In [22]:
model_rois = yearly_returns(result['model'].returns)
baseline_rois = yearly_returns(baseline_returns)

print("model rois =", model_rois)
print("baseline rois =", baseline_rois)

model rois = {2021: -0.06207058980218061, 2022: -0.1384337768484145, 2023: 0.8756319450754586, 2024: 0.23099676885419007, 'final': 0.8553334379009998, 'sharpe': 0.6218666510922056}
baseline rois = {2021: 0.009852934603402641, 2022: -0.0022877805890545444, 2023: 0.4123097495769479, 2024: 0.06037033817661208, 'final': 0.521652743494718, 'sharpe': 0.6013416324768195}


In [23]:
from evaluation.surge_metrics import compute_surge_metrics_from_buy_signals

metric = compute_surge_metrics_from_buy_signals(
    test_pred=test_pred,
    test_truth=test_truth,
    buy_signals=signals['buy_signals'],
    threshold=0.3
)

model_data = {
    'category': CATEGGORY,
    'baseline': 0,
    'precision': metric['actual_surge_rate_on_buy_signals'],
}
model_data.update(model_rois)

baseline_data = {
    'category': CATEGGORY,
    'baseline': 1,
    'precision': metric['overall_actual_surge_rate'],
}
baseline_data.update(baseline_rois)

result_df = pd.DataFrame([model_data, baseline_data])
result_df


,category,baseline,precision,2021,2022,2023,2024,final,sharpe
0,汽車,0,0.098119,-0.062071,-0.138434,0.875632,0.230997,0.855333,0.621867
1,汽車,1,0.055407,0.009853,-0.002288,0.412310,0.060370,0.521653,0.601342


In [24]:
model_df = pd.DataFrame([model_data])
baseline_df = pd.DataFrame([baseline_data])

display_df = pd.DataFrame(index=model_df.index)
display_df['category'] = model_df['category']

for col in model_df.columns:
    if col == 'category':
        continue
    model_values = model_df[col]
    baseline_values = baseline_df[col]
    delta = model_values - baseline_values
    if str(col) in ['2021', '2022', '2023', '2024', 'final']:
        display_df[col] = model_values.mul(100).round(1).apply(lambda x: f"{x:.0f}%") + \
                        delta.mul(100).round(1).apply(lambda x: f"({x:+.0f}%)")

    else:
        display_df[col] = model_values.round(2).astype(str) + \
                            delta.round(2).apply(lambda x: f" ({x:+.2f})")
                            
display_df = display_df.drop(columns=['baseline'])
display_df

,category,precision,2021,2022,2023,2024,final,sharpe
0,汽車,0.1 (+0.04),-6%(-7%),-14%(-14%),88%(+46%),23%(+17%),86%(+33%),0.62 (+0.02)
